# functional-module-wrap — worked example 2: Parametric F.dropout wrap honoring training mode

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `functional-module-wrap`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A parametric functional wrap stores its hyperparameters as plain Python attributes (not `nn.Parameter`) and forwards them to the functional. Dropout additionally must pass `self.training` so the module is a no-op in eval mode — the functional needs to be told the current mode explicitly.

## Worked solution

We wrap `F.dropout` into a `MyDropout` module that respects `train`/`eval`.

1. `__init__(p=0.5)` stores `self.p = p` as a plain attribute — it is configuration, not a learnable parameter, so it must not be an `nn.Parameter`.
2. `forward(x)` calls `F.dropout(x, p=self.p, training=self.training)`. The crucial argument is `training=self.training`: `nn.Module` flips `self.training` when you call `.train()` or `.eval()`, and the functional uses it to decide whether to actually drop.
3. In eval mode `F.dropout` becomes the identity, so the output equals the input exactly.
4. We add `extra_repr` so `print(model)` shows `p=...`. We verify: eval mode is identity, and there are zero parameters.

In [ ]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(1)

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        return F.dropout(x, p=self.p, training=self.training)

    def extra_repr(self):
        return f'p={self.p}'

x = t.randn(4, 4)
m = MyDropout(0.5)
m.eval()
print('eval is identity:', bool(t.allclose(m(x), x)))
print('num params:', sum(1 for _ in m.parameters()))
print(m)